In [ ]:
# Install required packages
!pip install transformers datasets peft accelerate bitsandbytes sentencepiece torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 MB 10.6 MB/s eta 0:00:00


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving merged_finetune_data.jsonl to merged_finetune_data.jsonl


# Set Hugging Face token for authentication
huggingface_token = "hf_QxnjCCfZtjOCFyQZOAQAyiKSGYIFTP"
huggingface_hub.login(token=huggingface_token)

# 1. Load Dataset:

In [ ]:
from datasets import load_dataset
dataset = load_dataset("json", data_files="merged_finetune_data.jsonl")
print("\nSample data:")
print(dataset["train"][0])

Generating train split: 0 examples [00:00, ? examples/s]


Sample data:
{'prompt': 'What is India according to the Union and its Territory?', 'completion': 'India, that is Bharat, shall be a Union of States.'}


Observations:

total records : 14543

format: {'prompt': ______ , 'completion':______}

# 2: load base model:

In [ ]:
# ===== Hugging Face Authentication =====
from huggingface_hub import notebook_login

# Run this to log in (it will open a popup for your Hugging Face token)
notebook_login()


Token has not been saved to git credential helper.


In [ ]:
from huggingface_hub import whoami

try:
    info = whoami()
    print("Logged in as:", info["name"])
except Exception as e:
    print("Not authenticated:", e)


✅ Logged in as: SaiPraneethTarlana


In [ ]:
from transformers import AutoModelForCausalLM

model_name = "meta-llama/Llama-2-7b-hf"

# Load the model in 4-bit precision for Colab-friendly training
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",      # Automatically use GPU if available
    load_in_4bit=True,      # Use 4-bit quantization (saves VRAM)
    torch_dtype="auto"      # Automatically select best precision
)

print("Model loaded successfully!")


config.json:   0%|          | 0.00/609 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!
The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


model.safetensors.index.json:   0%|          | 0.00/26.8k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/3.50G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/9.98G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/188 [00:00<?, ?B/s]

Model loaded successfully!


# 3: Prepare the model for LoRA fine-tuning using PEFT:

In [ ]:
from peft import LoraConfig, get_peft_model

# ===== LoRA Configuration =====
lora_config = LoraConfig(
    r=8,                   # LoRA rank (low-rank dimension)
    lora_alpha=16,         # Scaling factor
    target_modules=["q_proj", "v_proj"],  # Adapter layers to tune
    lora_dropout=0.05,     # Dropout for regularization
    bias="none",
    task_type="CAUSAL_LM"  # Language modeling task
)

# ===== Apply LoRA to the Base Model =====
model = get_peft_model(model, lora_config)

# ===== Check Trainable Parameters =====
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"Trainable Parameters: {trainable_params} / Total Parameters: {total_params}")


Trainable Parameters: 4194304 / Total Parameters: 3504607232


# 4: Load the Tokenizer:

In [ ]:
# ===== Import AutoTokenizer =====
from transformers import AutoTokenizer

# ===== Load Tokenizer =====
model_name = "meta-llama/Llama-2-7b-hf"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# ===== Check tokenizer configuration =====
print("✅ Tokenizer loaded successfully!")
print("Vocabulary size:", tokenizer.vocab_size)
print("Special tokens:", tokenizer.special_tokens_map)


tokenizer_config.json:   0%|          | 0.00/776 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

✅ Tokenizer loaded successfully!
Vocabulary size: 32000
Special tokens: {'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>'}


In [ ]:
# ===== Function to tokenize each example =====
def tokenize_function(example):
    # Encode prompt
    prompt_ids = tokenizer(example["prompt"], truncation=True, max_length=256)["input_ids"]
    # Encode completion
    completion_ids = tokenizer(example["completion"], truncation=True, max_length=256)["input_ids"]

    # Combine input + output for causal LM
    input_ids = prompt_ids + completion_ids
    # Labels are same as input_ids for causal LM
    return {"input_ids": input_ids, "labels": input_ids}

# ===== Apply tokenizer to dataset =====
tokenized_dataset = dataset.map(tokenize_function, batched=False)

# ===== Optional: Check sample =====
print(tokenized_dataset["train"][0])


Map:   0%|          | 0/14543 [00:00<?, ? examples/s]

{'prompt': 'What is India according to the Union and its Territory?', 'completion': 'India, that is Bharat, shall be a Union of States.', 'input_ids': [1, 1724, 338, 7513, 5034, 304, 278, 7761, 322, 967, 19833, 706, 29973, 1, 7513, 29892, 393, 338, 350, 8222, 271, 29892, 4091, 367, 263, 7761, 310, 3900, 29889], 'labels': [1, 1724, 338, 7513, 5034, 304, 278, 7761, 322, 967, 19833, 706, 29973, 1, 7513, 29892, 393, 338, 350, 8222, 271, 29892, 4091, 367, 263, 7761, 310, 3900, 29889]}


In [ ]:
# ===== Import TrainingArguments =====
from transformers import TrainingArguments, DataCollatorForSeq2Seq

# ===== Define Training Arguments =====
training_args = TrainingArguments(
    output_dir="./llama2-lora-finetuned",  # Where to save checkpoints & final model
    per_device_train_batch_size=2,         # Batch size per GPU/CPU
    gradient_accumulation_steps=4,         # To simulate a larger batch size
    learning_rate=2e-4,                    # Learning rate
    num_train_epochs=3,                     # Number of epochs (adjust based on dataset size)
    logging_steps=10,                       # Log loss every 10 steps
    save_strategy="steps",                  # Save checkpoint every N steps
    save_steps=200,                         # Save checkpoint every 200 steps
    fp16=True,                              # Mixed precision for faster training & less VRAM
    push_to_hub=True,                        # Push trained model to Hugging Face Hub
    report_to="none"                        # Disable unnecessary logging
)

# ===== Data collator for causal LM =====
data_collator = DataCollatorForSeq2Seq(tokenizer, return_tensors="pt")

print("✅ Training arguments are ready!")


✅ Training arguments are ready!


In [ ]:
# ===== Import Trainer =====
from transformers import Trainer

# ===== for LLaMA tokenizer padding =====
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    print("✅ Pad token was missing. Set pad_token = eos_token")

# ===== Initialize the Trainer =====
trainer = Trainer(
    model=model,                        # LoRA-wrapped model
    train_dataset=tokenized_dataset['train'],  # Training dataset
    tokenizer=tokenizer,                 # Tokenizer
    data_collator=data_collator,         # Prepares batches
    args=training_args                   # Training arguments
)


# ===== Start Training =====
trainer.train()


✅ Pad token was missing. Set pad_token = eos_token


/tmp/ipython-input-3612603039.py:10: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Step,Training Loss
10,2.574900
20,2.051200
30,1.698700
40,1.612300
50,1.499500
60,1.434900
70,1.467300
80,1.488500
90,1.383900
100,1.466400


In [ ]:
# ===== Step 8: Push Fine-Tuned Model to Hugging Face =====
hub_model_id = "SaiPraneethTarlana/llama2-lora-finetuned-LAWCHATBOT"

# Push directly from Trainer (no local save needed)
trainer.push_to_hub(hub_model_id)
print(f"✅ Model pushed to Hugging Face Hub: https://huggingface.co/{hub_model_id}")
